## Setting up the Spark Session

In modern Spark versions, SparkSession is the unified entry point and already encapsulates both SparkContext and SQLContext

In [31]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("CitiBikeNYCAnalysis") \
    .config("spark.executor.memory", "1gb") \
    .getOrCreate()

## Loading the  Citi Bike Trip Data

In [32]:
df = spark.read.option("header", True).csv("data/*.csv")

## Dataset Overview

In [33]:
df.printSchema()
df.count()
df.show(5, truncate=False)

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)

+----------------+-------------+-----------------------+-----------------------+-----------------------------+----------------+-----------------------+--------------+-----------------+------------------+------------------+------------------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name           |start_station_id|end_station_name       |end_station_id|start_lat

### Timestamp Conversion
#### For time-based analysis (e.g. trip duration, hourly demand patterns), the timestamp columns are converted from strings to Spark timestamp types.

In [34]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "started_at",
    to_timestamp("started_at")
).withColumn(
    "ended_at",
    to_timestamp("ended_at")
)

### Bike Type Distribution
#### As a first sanity check, we examine the distribution of bike types in the dataset
This allows us to verify that both electric and classic bikes are present and provides an initial indication of usage patterns.

In [35]:
df.groupBy("rideable_type").count().show()

+-------------+-------+
|rideable_type|  count|
+-------------+-------+
|electric_bike|3008344|
| classic_bike|1317209|
+-------------+-------+



### User Type Distribution
#### Next, we analyze the distribution of user types (members vs. casual riders)
This distinction is important for later analyses related to user behavior, trip duration, and system demand.

In [36]:
df.groupBy("member_casual").count().show()

+-------------+-------+
|member_casual|  count|
+-------------+-------+
|       casual| 841628|
|       member|3483925|
+-------------+-------+



## Data Cleaning

### Create trip duration

In order to calculate trip duration, we need to subtract the trip start time from the trip end time.  
Although `started_at` and `ended_at` are stored as timestamp columns in Spark, timestamps cannot be reliably subtracted directly for numerical analysis.

Spark handles timestamps as complex types, and subtracting them results in an interval data type, which is inconvenient for filtering, aggregation, and machine-learning models.

To avoid this issue, both timestamps are converted into Unix time using `unix_timestamp()`.

In [37]:
from pyspark.sql.functions import col, unix_timestamp

df = df.withColumn(
    "trip_duration_sec",
    unix_timestamp("ended_at") - unix_timestamp("started_at")
)

In [38]:
df.select("trip_duration_sec").show()

+-----------------+
|trip_duration_sec|
+-----------------+
|              294|
|              489|
|              752|
|              579|
|              449|
|             1146|
|              623|
|              946|
|              362|
|             1285|
|             1988|
|              115|
|             1233|
|              182|
|              172|
|             1044|
|              527|
|              469|
|              313|
|              817|
+-----------------+
only showing top 20 rows



### Remove obvious bad data (e.g. false starts < 60s)

In [39]:
df = df.filter(col("trip_duration_sec") >= 60)

### Create time features

In [40]:
from pyspark.sql.functions import hour, dayofweek, month

df = df.withColumn("start_hour", hour("started_at")) \
       .withColumn("start_day", dayofweek("started_at")) \
       .withColumn("start_month", month("started_at"))

In [41]:
df.groupBy("start_hour").count().orderBy("start_hour").show()

+----------+------+
|start_hour| count|
+----------+------+
|         0| 62146|
|         1| 36396|
|         2| 23024|
|         3| 15316|
|         4| 14139|
|         5| 29939|
|         6| 83818|
|         7|174956|
|         8|258168|
|         9|214263|
|        10|190669|
|        11|210602|
|        12|229394|
|        13|245324|
|        14|274419|
|        15|299971|
|        16|323582|
|        17|386725|
|        18|368594|
|        19|289193|
+----------+------+
only showing top 20 rows



### Create station-level flows (Objective A foundation)

Because .count() is a Spark aggregation function, and Spark always names its output column count by default, so we rename it to "start". 

In [42]:
starts = df.groupBy("start_station_id").count() \
           .withColumnRenamed("count", "starts")

ends = df.groupBy("end_station_id").count() \
         .withColumnRenamed("count", "ends")


Every trip has both a start and an end, but after aggregating trips by station, some stations may appear only as starting points or only as ending points. Using an outer join ensures that all stations are kept when computing net flow.

In [43]:
flows = starts.join(
    ends,
    starts.start_station_id == ends.end_station_id,
    how="outer"
).fillna(0).withColumn("net_flow", col("starts") - col("ends"))

### Distnace Features 

In [54]:
# we will include that we used AI for computing this

An approximate trip distance is computed using the geographic coordinates of the start and end stations. 

Because the Earth is round, latitude and longitude represent angular measurements rather than linear distances, and simple subtraction of coordinates would lead to incorrect results.  
Therefore, a standard geographic formula known as the **haversine formula** is used to estimate the straight-line distance between two points on the Earth’s surface.  
Although this does not capture the exact route taken, it provides a meaningful insights for trip length that can be reused in analysis and modeling.


### Haversine formula

For two points with latitude and longitude $\left(\phi_1, \lambda_1\right)$ and $\left(\phi_2, \lambda_2\right)$ (in **radians**):

$$
\begin{aligned}
a &= \sin^2\!\left(\frac{\phi_2 - \phi_1}{2}\right)
\;+\; \cos(\phi_1)\cos(\phi_2)\sin^2\!\left(\frac{\lambda_2 - \lambda_1}{2}\right) \\
c &= 2 \cdot \arctan2(\sqrt{a}, \sqrt{1-a}) \\
d &= R \cdot c
\end{aligned}
$$

where: 
- d = distance between the two points
- R = Earth’s radius (≈ 6371 km)
- ϕ = latitude
- λ = longitude


In [52]:
from pyspark.sql.functions import radians, sin, cos, sqrt, atan2

R = 6371  # Earth radius in kilometers

df = df.withColumn(
    "distance_km",
    2 * R * atan2(
        sqrt(
            sin((radians(df.end_lat) - radians(df.start_lat)) / 2)**2 +
            cos(radians(df.start_lat)) * cos(radians(df.end_lat)) *
            sin((radians(df.end_lng) - radians(df.start_lng)) / 2)**2
        ),
        sqrt(
            1 - (
                sin((radians(df.end_lat) - radians(df.start_lat)) / 2)**2 +
                cos(radians(df.start_lat)) * cos(radians(df.end_lat)) *
                sin((radians(df.end_lng) - radians(df.start_lng)) / 2)**2
            )
        )
    )
)

In [55]:
df.select("distance_km").summary().show()

+-------+------------------+
|summary|       distance_km|
+-------+------------------+
|  count|           4311393|
|   mean| 2.061715206029513|
| stddev|1.7585701825985094|
|    min|               0.0|
|    25%|0.8757170557211219|
|    50%|1.5447855404223063|
|    75%| 2.700947029331631|
|    max| 28.32155976817476|
+-------+------------------+



In [56]:
df.cache()

DataFrame[ride_id: string, rideable_type: string, started_at: timestamp, ended_at: timestamp, start_station_name: string, start_station_id: string, end_station_name: string, end_station_id: string, start_lat: string, start_lng: string, end_lat: string, end_lng: string, member_casual: string, trip_duration_sec: bigint, start_hour: int, start_day: int, start_month: int, distance_km: double]

#### Saving the cleaned data to disk allows the preprocessing steps to be reused without reloading and cleaning the raw CSV files each time, which is costly for large datasets.

In [57]:
df.write.mode("overwrite").parquet("processed/clean_trips")

#### !!Note: please do not run `spark.stop()` before the previous cell has exeuted completely (the dataset has been successfully written to disk)

In [58]:
spark.stop()